# 06 — Before / After Analysis

Compares the original trained YOLOv3 against the Tucker-compressed version:
parameter count, model size, latency, mAP@0.5, and side-by-side detections.

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath("../src"))
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from model import YOLOv3
from dataset import CocoSubsetDataset, yolo_collate_fn
from tucker_pipeline import apply_selected_ranks
from postprocess import predict_boxes
from tucker_phase1 import evaluate_map
from analysis_utils import count_params, model_size_mb, measure_latency

with open("../checkpoints/run_config.json") as f:
    cfg = json.load(f)
CLASS_NAMES, IMG_SIZE, DATA_ROOT, BATCH_SIZE = cfg["class_names"], cfg["img_size"], cfg["data_root"], cfg["batch_size"]
NUM_CLASSES = len(CLASS_NAMES)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open("../checkpoints/compression_selection.json") as f:
    sel_info = json.load(f)
selected_ranks = sel_info["selected_ranks"]

## Load both models

In [ ]:
state = torch.load("../checkpoints/yolov3_best.pt", map_location="cpu")
original = YOLOv3(num_classes=NUM_CLASSES)
original.load_state_dict(state["model_state"])
original.to(DEVICE).eval()

compressed = YOLOv3(num_classes=NUM_CLASSES)
compressed.load_state_dict(state["model_state"])
compressed = apply_selected_ranks(compressed, selected_ranks, DEVICE)
compressed.load_state_dict(torch.load("../checkpoints/yolov3_compressed.pt", map_location="cpu"))
compressed.to(DEVICE).eval()
print("both models loaded")

## Params, size, latency

In [ ]:
p_o, p_c = count_params(original), count_params(compressed)
s_o, s_c = model_size_mb(original), model_size_mb(compressed)
l_o = measure_latency(original, (1,3,IMG_SIZE,IMG_SIZE), DEVICE)
l_c = measure_latency(compressed, (1,3,IMG_SIZE,IMG_SIZE), DEVICE)

print(f"{'':16s}{'original':>14s}{'compressed':>14s}{'change':>12s}")
print(f"{'params':16s}{p_o:>14,d}{p_c:>14,d}{(1-p_c/p_o)*100:>11.1f}%")
print(f"{'size (MB)':16s}{s_o:>14.1f}{s_c:>14.1f}{(1-s_c/s_o)*100:>11.1f}%")
print(f"{'latency (ms)':16s}{l_o:>14.2f}{l_c:>14.2f}{(1-l_c/l_o)*100:>+11.1f}%")

Latency note: Tucker replaces one conv with three sequential convs, so wall-clock speed doesn't always improve in proportion to parameter reduction — the win here is model size / memory footprint. State this explicitly if asked.

## mAP@0.5 comparison

In [ ]:
val_ds = CocoSubsetDataset(DATA_ROOT, "val2017", CLASS_NAMES, img_size=IMG_SIZE,
                            images_per_class=30, augment=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=yolo_collate_fn, num_workers=0)

map_o = evaluate_map(original, val_loader, NUM_CLASSES, DEVICE, IMG_SIZE, max_batches=None)
map_c = evaluate_map(compressed, val_loader, NUM_CLASSES, DEVICE, IMG_SIZE, max_batches=None)
print(f"original mAP@0.5:   {map_o:.4f}")
print(f"compressed mAP@0.5: {map_c:.4f}  ({map_c-map_o:+.4f})")

## Side-by-side detections on one image

In [ ]:
def draw(ax, img_t, boxes, scores, labels, title):
    ax.imshow(img_t.permute(1,2,0).cpu().numpy())
    for b, s, l in zip(boxes, scores, labels):
        x1,y1,x2,y2 = b.tolist()
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor="lime",linewidth=2))
        ax.text(x1, max(y1-4,0), f"{CLASS_NAMES[int(l)]} {s:.2f}", color="black",
                fontsize=8, bbox=dict(facecolor="lime", alpha=0.7, pad=1))
    ax.set_title(f"{title} ({len(boxes)} det)"); ax.axis("off")

img_t, _ = val_ds[0]
batch = img_t.unsqueeze(0).to(DEVICE)
with torch.no_grad():
    po = predict_boxes(original, batch, NUM_CLASSES, conf_thresh=0.5)[0]
    pc = predict_boxes(compressed, batch, NUM_CLASSES, conf_thresh=0.5)[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
draw(axes[0], img_t, *po, "Original")
draw(axes[1], img_t, *pc, "Compressed")
plt.tight_layout(); plt.show()

## Summary table

In [ ]:
import pandas as pd
pd.DataFrame({
    "metric": ["Total params", "Model size (MB)", "Latency (ms/img)", "mAP@0.5"],
    "original": [f"{p_o:,}", f"{s_o:.1f}", f"{l_o:.2f}", f"{map_o:.4f}"],
    "compressed": [f"{p_c:,}", f"{s_c:.1f}", f"{l_c:.2f}", f"{map_c:.4f}"],
    "change": [f"{(1-p_c/p_o)*100:.1f}%", f"{(1-s_c/s_o)*100:.1f}%",
               f"{(1-l_c/l_o)*100:+.1f}%", f"{map_c-map_o:+.4f}"],
})